# Runnable Log Streaming

The `log_stream.py` module defines the data structures and tracer used by LangChain's asynchronous run-log streaming interface.

Run activity is represented as JSON Patch operations. Consumers may process each `RunLogPatch` incrementally or combine patches into a `RunLog` containing the reconstructed state of the root run and its included sub-runs.

# LogEntry

`LogEntry` defines the state recorded for one included sub-run.

## Bases

- `TypedDict`

## Attributes

1. `id`: Stores the unique identifier of the sub-run.
   * **Type:**
     ```python
     id: str
     ```

2. `name`: Stores the name of the object being executed.
   * **Type:**
     ```python
     name: str
     ```

3. `type`: Stores the run type, such as `"prompt"`, `"chain"`, or `"llm"`.
   * **Type:**
     ```python
     type: str
     ```

4. `tags`: Stores the tags associated with the run.
   * **Type:**
     ```python
     tags: list[str]
     ```

5. `metadata`: Stores key-value metadata associated with the run.
   * **Type:**
     ```python
     metadata: dict[str, Any]
     ```

6. `start_time`: Stores the run's start time as an ISO 8601 string.
   * **Type:**
     ```python
     start_time: str
     ```

7. `streamed_output_str`: Stores text tokens or structured token values streamed by an LLM run.
   * **Type:**
     ```python
     streamed_output_str: list[str]
     ```

8. `streamed_output`: Stores output chunks streamed by the run.
   * **Type:**
     ```python
     streamed_output: list[Any]
     ```

9. `inputs`: Optionally stores the standardized input supplied to the run.

   This field is included when the internal `"streaming_events"` schema format is used.

   * **Type:**
     ```python
     inputs: NotRequired[
         Any | None
     ]
     ```

10. `final_output`: Stores the completed output after the run finishes successfully.
    * **Type:**
      ```python
      final_output: Any | None
      ```

11. `end_time`: Stores the run's completion time as an ISO 8601 string.

    The value remains `None` until the run finishes.

    * **Type:**
      ```python
      end_time: str | None
      ```

# RunState

`RunState` defines the reconstructed state of the root run and its included sub-runs.

## Bases

- `TypedDict`

## Attributes

1. `id`: Stores the unique identifier of the root run.
   * **Type:**
     ```python
     id: str
     ```

2. `streamed_output`: Stores output chunks yielded by the root Runnable.
   * **Type:**
     ```python
     streamed_output: list[Any]
     ```

3. `final_output`: Stores the current aggregated output of the root run.

   The value is updated as streamed chunks are received when the output type supports addition.

   * **Type:**
     ```python
     final_output: Any | None
     ```

4. `name`: Stores the name of the root Runnable.
   * **Type:**
     ```python
     name: str
     ```

5. `type`: Stores the type of the root run.
   * **Type:**
     ```python
     type: str
     ```

6. `logs`: Maps generated log keys to included sub-run entries.

   When filters are configured, only matching sub-runs appear in this mapping.

   * **Type:**
     ```python
     logs: dict[
         str,
         LogEntry
     ]
     ```

# RunLogPatch

`RunLogPatch` stores one or more JSON Patch operations describing an incremental change to a run log.

Applying the accumulated operations to an empty state reconstructs the current `RunState`.

## Attributes

1. `ops`: Stores the JSON Patch operations contained in the patch.
   * **Type:**
     ```python
     ops: list[
         dict[str, Any]
     ]
     ```

### Methods

1. `__init__`: Creates a run-log patch from one or more JSON Patch operations.
   * **Syntax:**
     ```python
     __init__(
         self,
         *ops: dict[str, Any] # JSON Patch operations
     ) -> None
     ```

2. `__add__`: Combines this patch with another `RunLogPatch` and reconstructs a complete `RunLog`.

   The operations from both patches are applied to an initially empty state. A `TypeError` is raised when the right-hand value is not exactly a `RunLogPatch`.

   * **Syntax:**
     ```python
     __add__(
         self,
         other: RunLogPatch | Any # Patch to append
     ) -> RunLog
     ```

3. `__repr__`: Returns a formatted representation of the patch operations.
   * **Syntax:**
     ```python
     __repr__(
         self
     ) -> str
     ```

4. `__eq__`: Compares two patches by their operation lists.
   * **Syntax:**
     ```python
     __eq__(
         self,
         other: object # Object to compare
     ) -> bool
     ```

## Hashing

`RunLogPatch` instances are mutable and unhashable.

# RunLog

`RunLog` extends `RunLogPatch` with the current reconstructed `RunState`.

Adding another patch applies only the new operations to the existing state and returns a new `RunLog`.

## Bases

- `RunLogPatch`

## Attributes

1. `state`: Stores the current state produced by applying all accumulated operations.
   * **Type:**
     ```python
     state: RunState
     ```

### Methods

1. `__init__`: Creates a run log from accumulated operations and an initial reconstructed state.
   * **Syntax:**
     ```python
     __init__(
         self,
         *ops: dict[str, Any], # Accumulated JSON Patch operations
         state: RunState # Current reconstructed run state
     ) -> None
     ```

2. `__add__`: Applies another `RunLogPatch` to the current state and returns a new `RunLog`.

   A `TypeError` is raised when the right-hand value is not exactly a `RunLogPatch`.

   * **Syntax:**
     ```python
     __add__(
         self,
         other: RunLogPatch | Any # Patch to apply
     ) -> RunLog
     ```

3. `__repr__`: Returns a formatted representation of the reconstructed state.
   * **Syntax:**
     ```python
     __repr__(
         self
     ) -> str
     ```

4. `__eq__`: Compares two run logs by both reconstructed state and accumulated operations.
   * **Syntax:**
     ```python
     __eq__(
         self,
         other: object # Object to compare
     ) -> bool
     ```

## Hashing

`RunLog` instances are mutable and unhashable.

# LogStreamCallbackHandler

`LogStreamCallbackHandler` is a tracer that converts run lifecycle activity into an asynchronous stream of `RunLogPatch` objects.

The root run initializes the `RunState`. Matching child runs are added under `RunState["logs"]`. Output chunks, final outputs, inputs, and end times are subsequently delivered through JSON Patch operations.

## Bases

- `BaseTracer`
- `_StreamingCallbackHandler[Any]`

## Attributes

1. `auto_close`: Controls whether the patch stream closes automatically when the root run finishes.
   * **Type:**
     ```python
     auto_close: bool
     ```

2. `include_names`: Optionally restricts included child runs to matching names.
   * **Type:**
     ```python
     include_names: Sequence[str] | None
     ```

3. `include_types`: Optionally restricts included child runs to matching run types.
   * **Type:**
     ```python
     include_types: Sequence[str] | None
     ```

4. `include_tags`: Optionally restricts included child runs to runs containing at least one matching tag.
   * **Type:**
     ```python
     include_tags: Sequence[str] | None
     ```

5. `exclude_names`: Optionally removes child runs with matching names.
   * **Type:**
     ```python
     exclude_names: Sequence[str] | None
     ```

6. `exclude_types`: Optionally removes child runs with matching run types.
   * **Type:**
     ```python
     exclude_types: Sequence[str] | None
     ```

7. `exclude_tags`: Optionally removes child runs containing an excluded tag.
   * **Type:**
     ```python
     exclude_tags: Sequence[str] | None
     ```

8. `lock`: Stores the lock used while generating unique log keys for runs that share a name.
   * **Type:**
     ```python
     lock: threading.Lock
     ```

9. `send_stream`: Stores the sending side of the internal memory stream.
   * **Type:**
     ```python
     send_stream: Any
     ```

10. `receive_stream`: Stores the receiving side of the internal memory stream.
    * **Type:**
      ```python
      receive_stream: Any
      ```

11. `root_id`: Stores the identifier of the first run registered by the tracer.
    * **Type:**
      ```python
      root_id: UUID | None
      ```

### Methods

1. `__init__`: Creates a run-log streaming tracer.

   Include filters use OR logic across names, types, and tags. Exclude filters are then applied to the included result.

   The internal `_schema_format` accepts `"original"` or `"streaming_events"`. A `ValueError` is raised for another value.

   * **Syntax:**
     ```python
     __init__(
         self,
         *,
         auto_close: bool = True, # Close the stream when the root run finishes
         include_names: Sequence[str] | None = None, # Included run names
         include_types: Sequence[str] | None = None, # Included run types
         include_tags: Sequence[str] | None = None, # Included run tags
         exclude_names: Sequence[str] | None = None, # Excluded run names
         exclude_types: Sequence[str] | None = None, # Excluded run types
         exclude_tags: Sequence[str] | None = None, # Excluded run tags
         _schema_format: Literal[
             "original",
             "streaming_events"
         ] = "streaming_events" # Internal input and output schema format
     ) -> None
     ```

2. `__aiter__`: Returns an asynchronous iterator over generated log patches.
   * **Syntax:**
     ```python
     __aiter__(
         self
     ) -> AsyncIterator[
         RunLogPatch
     ]
     ```

3. `send`: Sends one `RunLogPatch` containing the supplied operations to the internal stream.

   The method returns `True` after successful submission. Stream errors are allowed to propagate.

   * **Syntax:**
     ```python
     send(
         self,
         *ops: dict[str, Any] # JSON Patch operations to send
     ) -> bool
     ```

4. `tap_output_aiter`: Observes an asynchronous output iterator and records each included child-run chunk.

   Root-run output is handled separately by the asynchronous log implementation. Runs excluded from the log are passed through without recording. Every original chunk is yielded unchanged.

   * **Syntax:**
     ```python
     async tap_output_aiter(
         self,
         run_id: UUID, # Identifier of the run producing output
         output: AsyncIterator[T] # Asynchronous output iterator to observe
     ) -> AsyncIterator[T]
     ```

5. `tap_output_iter`: Observes a synchronous output iterator and records each included child-run chunk.

   Root-run output is handled separately. Runs excluded from the log are passed through without recording. Every original chunk is yielded unchanged.

   * **Syntax:**
     ```python
     tap_output_iter(
         self,
         run_id: UUID, # Identifier of the run producing output
         output: Iterator[T] # Output iterator to observe
     ) -> Iterator[T]
     ```

6. `include_run`: Determines whether a child run should appear in the log.

   The root run is always excluded from `RunState["logs"]`. When no include filters are configured, child runs are included by default. Configured exclusion rules are applied after inclusion rules.

   * **Syntax:**
     ```python
     include_run(
         self,
         run: Run # Run to test against the filters
     ) -> bool
     ```

7. `_persist_run`: Implements the required tracer persistence hook.

   The method intentionally performs no action because run-log streaming uses incremental lifecycle hooks rather than persisting the completed run tree.

   * **Syntax:**
     ```python
     _persist_run(
         self,
         run: Run # Completed root run
     ) -> None
     ```

8. `_on_run_create`: Initializes the root state or adds one included child-run entry.

   The first observed run becomes the root and generates a root-state replacement patch. Child runs with duplicate names receive generated keys such as `<name>:2`.

   In the `"streaming_events"` schema format, the standardized run input is added to the child entry.

   * **Syntax:**
     ```python
     _on_run_create(
         self,
         run: Run # Newly created run
     ) -> None
     ```

9. `_on_run_update`: Adds the final input, final output, and end time for an included run.

   When the updated run is the root and `auto_close` is enabled, the sending stream is closed in a `finally` block.

   * **Syntax:**
     ```python
     _on_run_update(
         self,
         run: Run # Completed or updated run
     ) -> None
     ```

10. `_on_llm_new_token`: Adds token-oriented and output-oriented patches for an included LLM or chat-model run.

    `streamed_output_str` receives the token value. For a `ChatGenerationChunk`, `streamed_output` receives its message; otherwise it receives the token.

    * **Syntax:**
      ```python
      _on_llm_new_token(
          self,
          run: Run, # Run receiving the token
          token: str
          | list[
              str
              | dict[str, Any]
          ], # Token or structured content blocks
          chunk: GenerationChunk
          | ChatGenerationChunk
          | None # Optional generation chunk
      ) -> None
      ```

## Patch and State Streaming

The Runnable log interface may expose either of the following forms:

- `diff=True` yields incremental `RunLogPatch` objects.
- `diff=False` cumulatively applies the patches and yields complete `RunLog` objects.

Root output chunks may be stored in `RunState["streamed_output"]`. The current `final_output` is updated by calculating JSON Patch differences between the previous and newly aggregated output.